In [1]:
from transformers import pipeline

# load the zero-shot classifier (first run downloads the model — may take a minute)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["opportunity", "risk", "trend"]

text = "Lufthansa faces pilot strikes and rising fuel costs threatening its profits."
result = classifier(text, candidate_labels=labels)

print(result)

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'sequence': 'Lufthansa faces pilot strikes and rising fuel costs threatening its profits.', 'labels': ['risk', 'trend', 'opportunity'], 'scores': [0.9322348237037659, 0.05446822941303253, 0.013296927325427532]}


In [2]:
import json
documents = json.load(open("data/lufthansa_data.json", encoding="utf-8"))

labels = ["opportunity", "risk", "trend"]

# test on the first 5 real docs
for d in documents[:5]:
    result = classifier(d["text"], candidate_labels=labels)
    cat   = result["labels"][0]
    score = result["scores"][0]
    print(f"[{cat}] ({score:.2f})  {d['text'][:90]}")
    print()

[trend] (0.53)  Lufthansa Group releases its third-quarter 2025 financial. Travel Radar - Aviation News > 

[trend] (0.76)  Lufthansa Group Financial Results: More Passengers, Less. This morning Lufthansa Group pos

[trend] (0.65)  Lufthansa publishes financial results for Q1 2020 - InsideFlyer. Lufthansa publishes finan

[trend] (0.64)  Lufthansa records best-ever nine-month result | News | Breaking. As a result, the Lufthans

[trend] (0.68)  Lufthansa reveals improved 2023 financial figures. The strong result for the financial yea



In [3]:
sentiment_pipe = pipeline("sentiment-analysis",
                          model="cardiffnlp/twitter-roberta-base-sentiment-latest")

#3-class sentiment (negative / neutral / positive)
from collections import Counter

counts = Counter()
for d in documents:
    label = sentiment_pipe(d["text"][:512])[0]["label"]
    d["sentiment"] = label          # store it on the doc (reuse later — no re-running)
    counts[label] += 1

print(counts)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Counter({'neutral': 207, 'positive': 95, 'negative': 39})


In [4]:
labels = ["opportunity", "risk", "trend"]
sev_labels = ["high severity", "medium severity", "low severity"]

for d in documents:
    result = classifier(d["text"], candidate_labels=labels)   # call once
    d["category"]       = result["labels"][0]                 # top label
    d["category_score"] = round(float(result["scores"][0]), 3)  # confidence
    if d["category"] == "risk":                                 # severity only for risks
        d["severity"] = classifier(d["text"], candidate_labels=sev_labels)["labels"][0].split()[0].capitalize()

print("Done classifying")
from collections import Counter
print(Counter(d["category"] for d in documents))
print(Counter(d["severity"] for d in documents if d["category"] == "risk"))

Done classifying
Counter({'trend': 165, 'risk': 154, 'opportunity': 22})
Counter({'High': 89, 'Medium': 62, 'Low': 3})


In [5]:
import json
json.dump(documents,
          open("data/lufthansa_labeled.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saved", len(documents), "labeled docs")

Saved 341 labeled docs
